# 03 — Embeddings
**Goal:** Show that sentence embeddings capture semantic similarity between startups.
This is what powers the ChromaDB comparables search.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv('../data/sample/startups_sample.csv')
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df['description'].tolist(), show_progress_bar=True)
print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
# PCA to 2D — visualize sector clustering
pca = PCA(n_components=2)
coords = pca.fit_transform(embeddings)

sectors = df['sector'].unique()
colors = plt.cm.tab20(np.linspace(0, 1, len(sectors)))
sector_color = {s: c for s, c in zip(sectors, colors)}

plt.figure(figsize=(12, 7))
for sector in sectors:
    mask = df['sector'] == sector
    plt.scatter(coords[mask, 0], coords[mask, 1],
                label=sector, color=sector_color[sector], s=80, alpha=0.8)

plt.title('Startup Embeddings — PCA 2D\nClusters show semantic similarity by sector',
          fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('../data/sample/eda_embeddings_pca.png', dpi=120)
plt.show()

In [ ]:
# Semantic similarity search — what ChromaDB does at query time
query = 'AI platform for climate risk and carbon accounting'
query_emb = model.encode([query])
sims = cosine_similarity(query_emb, embeddings)[0]
top5 = np.argsort(sims)[::-1][:5]

print(f'Query: "{query}"')
print('\nTop 5 similar startups:')
for i, idx in enumerate(top5):
    print(f'  {i+1}. {df.iloc[idx]["name"]} ({df.iloc[idx]["sector"]}) — similarity: {sims[idx]:.3f}')